# Candidate-Job Fit Evaluation System

This notebook implements a **multi-agent system** using LangGraph to evaluate the compatibility (fit) between enriched candidate profiles and enriched job offers.

## Architecture Overview

### Hybrid Collaborative-Supervisor Pattern

The system uses a three-phase approach:

1. **Phase 1: Parallel Specialized Analysis** (Swarm)
   - Sector Alignment Agent
   - Skills Matching Agent
   - Seniority Alignment Agent
   - Competencies Matching Agent
   - Red Flags Detector Agent

2. **Phase 2: Domain Expert Analysis** (Conditional)
   - Dynamically instantiated based on job sector
   - Provides sector-specific deep evaluation

3. **Phase 3: Orchestration & Synthesis** (Supervisor)
   - Integrates all evaluations
   - Applies dynamic weighting
   - Produces final verdict with full explainability

### Key Features

- **Sector-agnostic**: Works across all industries
- **Semantic disambiguation**: Resolves ambiguities using enrichment data
- **Dynamic weighting**: Adapts scoring based on sector and seniority
- **Full explainability**: Every decision traced to evidence
- **Red flag detection**: Critical disqualifiers identified early
- **Conditional routing**: Early termination or domain expert activation

In [ ]:
# Standard library imports
import json
import operator
from datetime import datetime
from pathlib import Path
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

# Third-party imports
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

## State Schema Definition

The state flows through all agents and accumulates evaluations.

In [ ]:
%%writefile ../src/candidate_job_evaluation/state.py"""State schema for candidate-job evaluation system."""import operatorfrom datetime import datetimefrom typing import Annotated, Any, Dict, List, Literal, Optional, TypedDictfrom langchain_core.messages import BaseMessagefrom pydantic import BaseModel, Fieldclass AgentEvaluation(BaseModel):    """Output from a specialized evaluation agent."""    agent_name: str = Field(description="Name of the agent")    score: int = Field(ge=0, le=100, description="Score from 0-100")    alignment_level: Literal[        "perfect_match", "strong_match", "partial_match", "weak_match", "mismatch"    ] = Field(description="Categorical alignment level")    reasoning: str = Field(description="Step-by-step reasoning (Chain-of-Thought)")    evidence: List[str] = Field(description="Evidence points from inputs")    red_flags: List[str] = Field(        default_factory=list, description="Identified red flags"    )    confidence: float = Field(        ge=0.0, le=1.0, description="Confidence in evaluation"    )class FinalEvaluation(BaseModel):    """Final synthesized evaluation."""    overall_score: int = Field(ge=0, le=100, description="Weighted overall score")    recommendation: Literal[        "highly_recommend", "recommend", "acceptable", "not_recommend"    ] = Field(description="Final recommendation")    summary: str = Field(description="Executive summary")    strengths: List[str] = Field(description="Key strengths identified")    concerns: List[str] = Field(description="Key concerns identified")    missing_skills: List[str] = Field(        description="Critical skills candidate is missing"    )    all_red_flags: List[str] = Field(description="All red flags collected")    agent_scores: Dict[str, int] = Field(        description="Individual agent scores for transparency"    )    decision_tree: str = Field(description="Human-readable decision explanation")class EvaluationState(TypedDict):    """State that flows through the evaluation graph."""    # Inputs    candidate: Dict[str, Any]    job_offer: Dict[str, Any]    # Agent evaluations (accumulated)    sector_evaluation: Optional[AgentEvaluation]    skills_evaluation: Optional[AgentEvaluation]    seniority_evaluation: Optional[AgentEvaluation]    competencies_evaluation: Optional[AgentEvaluation]    red_flags_evaluation: Optional[AgentEvaluation]    domain_expert_evaluation: Optional[AgentEvaluation]    # Final synthesis    final_evaluation: Optional[FinalEvaluation]    # Control flags    critical_red_flags_found: bool    early_termination: bool    # Metadata    evaluation_id: str    timestamp: str    messages: Annotated[List[BaseMessage], operator.add]

## Configuration

Flexible configuration for LLM, thresholds, and weights.

In [ ]:
%%writefile ../src/candidate_job_evaluation/config.py"""Configuration for evaluation system."""from typing import Dict, List, Literalfrom pydantic import BaseModel, Fieldclass EvaluationConfig(BaseModel):    """Configuration for candidate-job evaluation system."""    # LLM settings    llm_provider: Literal["openai"] = "openai"    llm_model: str = "gpt-4o"    llm_temperature: float = Field(        default=0.1, description="Low temperature for consistency"    )    # Scoring thresholds    highly_recommend_threshold: int = 85    recommend_threshold: int = 70    acceptable_threshold: int = 55    # Default agent weights (sector-agnostic baseline)    default_weights: Dict[str, float] = Field(        default={            "sector": 0.25,            "skills": 0.30,            "seniority": 0.15,            "competencies": 0.20,            "domain_expert": 0.10,        }    )    # Sector-specific weight adjustments    sector_weight_adjustments: Dict[str, Dict[str, float]] = Field(        default={            "software": {"skills": 0.40, "competencies": 0.15},            "construction": {"sector": 0.30, "competencies": 0.25, "skills": 0.20},            "finance": {"competencies": 0.30, "skills": 0.25},        }    )    # Critical red flags (auto-disqualify)    critical_red_flag_types: List[str] = Field(        default=[            "sector_complete_mismatch",            "missing_mandatory_certification",            "insufficient_experience_years",        ]    )    # Error handling    agent_failure_default_score: int = 50    max_retries: int = 2

## Agent Prompts

Chain-of-Thought prompts for each specialized agent.

In [ ]:
%%writefile ../src/candidate_job_evaluation/prompts.py"""Prompts for specialized evaluation agents."""SECTOR_AGENT_PROMPT = """You are a Sector Alignment Agent evaluating candidate-job compatibility.**Inputs:**- Candidate sector: {candidate_sector}- Job sector: {job_sector}- Candidate disambiguation map: {candidate_disambiguation}- Job red flags: {job_red_flags}**Your task:**Evaluate sector alignment step-by-step:1. **Primary Sector Match**: Compare primary sectors directly2. **Secondary Sector Overlap**: Check if secondary sectors overlap3. **Transferability Analysis**: Assess if candidate's experience transfers to job sector4. **Disambiguation Verification**: Use disambiguation data to resolve ambiguities5. **Red Flags Detection**: Identify critical sector mismatches6. **Scoring**: Calculate 0-100 score based on alignment**Scoring Guidelines:**- 90-100: Perfect sector match (primary sectors identical)- 75-89: Strong match (primary matches or strong secondary overlap)- 60-74: Partial match (transferable skills exist)- 40-59: Weak match (limited transferability)- 0-39: Mismatch (incompatible sectors)**Critical Red Flags:**- Job lists "Experiencia limitada a [different sector]" as red flag- No evidence of sector-relevant work**Output format (JSON):**{{  "agent_name": "sector_alignment",  "score": <0-100>,  "alignment_level": "<perfect_match|strong_match|partial_match|weak_match|mismatch>",  "reasoning": "<step-by-step explanation>",  "evidence": [<list of evidence points>],  "red_flags": [<list of flags or empty>],  "confidence": <0.0-1.0>}}Think step by step and be precise."""SKILLS_AGENT_PROMPT = """You are a Skills Matching Agent evaluating candidate-job compatibility.**Inputs:**- Candidate skills (explicit): {candidate_skills_explicit}- Candidate skills (implicit): {candidate_skills_implicit}- Job required skills: {job_required_skills}- Job preferred skills: {job_preferred_skills}- Candidate synonym map: {candidate_synonyms}- Job must-have experience: {job_must_have}**Your task:**Evaluate skills match step-by-step:1. **Required Skills Match**: Calculate coverage of job's required skills2. **Preferred Skills Match**: Calculate coverage of job's preferred skills3. **Semantic Matching**: Use synonym maps to match equivalent skills4. **Skill Level Comparison**: Compare candidate's skill levels with job requirements5. **Gap Analysis**: Identify critical missing skills6. **Transferable Skills**: Identify implicit skills that transfer7. **Scoring**: Calculate weighted score**Scoring Guidelines:**- Required skills: 70% weight- Preferred skills: 30% weight- 100% required coverage + 100% preferred = 100 score- Missing required skills severely penalize score- Level mismatches (e.g., novice vs advanced required) reduce score**Critical Red Flags:**- Missing >2 required skills marked "must-have"- Core domain skills absent**Output format (JSON):**{{  "agent_name": "skills_matching",  "score": <0-100>,  "alignment_level": "<perfect_match|strong_match|partial_match|weak_match|mismatch>",  "reasoning": "<step-by-step explanation>",  "evidence": [<matched skills with levels>],  "red_flags": [<critical gaps>],  "confidence": <0.0-1.0>}}Think step by step. Use semantic matching extensively."""SENIORITY_AGENT_PROMPT = """You are a Seniority Alignment Agent evaluating candidate-job compatibility.**Inputs:**- Candidate seniority: {candidate_seniority}- Candidate experience years: {candidate_years}- Job required seniority: {job_seniority}- Job experience years: {job_years}- Candidate seniority evidence: {candidate_seniority_evidence}- Job red flags: {job_red_flags}**Your task:**Evaluate seniority alignment step-by-step:1. **Experience Years Match**: Compare years of experience2. **Seniority Level Match**: Compare seniority levels (junior/mid/senior/lead/executive)3. **Evidence Validation**: Check if candidate's evidence supports their seniority4. **Trajectory Analysis**: Assess if candidate's career trajectory fits job requirements5. **Red Flags Detection**: Identify over/under-qualification6. **Scoring**: Calculate alignment score**Seniority Level Mapping:**- junior: 0-2 years- mid: 2-5 years- senior: 5-8 years- lead: 8-12 years- principal/executive: 12+ years**Scoring Guidelines:**- Exact match: 95-100- One level difference (acceptable): 70-85- Two levels difference: 40-60- Over-qualified (candidate senior >> job junior): 50-70 (potential flight risk)- Under-qualified (candidate junior << job senior): 20-40**Critical Red Flags:**- Candidate has <50% of required experience years- Candidate is 2+ seniority levels below required**Output format (JSON):**{{  "agent_name": "seniority_alignment",  "score": <0-100>,  "alignment_level": "<perfect_match|strong_match|partial_match|weak_match|mismatch>",  "reasoning": "<step-by-step explanation>",  "evidence": [<evidence points>],  "red_flags": [<qualification issues>],  "confidence": <0.0-1.0>}}Think step by step."""COMPETENCIES_AGENT_PROMPT = """You are a Competencies Matching Agent evaluating candidate-job compatibility.**Inputs:**- Candidate competencies: {candidate_competencies}- Job required competencies: {job_competencies}- Candidate leadership indicators: {candidate_leadership}- Job leadership level: {job_leadership_level}**Your task:**Evaluate competencies match step-by-step:1. **Functional Competencies Match**: Compare technical/domain competencies2. **Soft Competencies Match**: Compare communication, teamwork, etc.3. **Managerial Competencies Match**: Compare leadership competencies4. **Competency Level Alignment**: Check if candidate's levels match job's requirements5. **Gap Analysis**: Identify missing competencies6. **Scoring**: Calculate weighted score**Competency Types:**- Functional (40% weight): Domain-specific abilities- Soft (30% weight): Interpersonal and communication- Managerial (30% weight): Leadership and strategic**Scoring Guidelines:**- Calculate coverage per competency type- Apply weights and aggregate- Penalize level mismatches (e.g., mid-level competency vs senior required)**Critical Red Flags:**- Missing >3 required functional competencies- No evidence of required managerial competencies for lead+ roles**Output format (JSON):**{{  "agent_name": "competencies_matching",  "score": <0-100>,  "alignment_level": "<perfect_match|strong_match|partial_match|weak_match|mismatch>",  "reasoning": "<step-by-step explanation>",  "evidence": [<matched competencies>],  "red_flags": [<critical gaps>],  "confidence": <0.0-1.0>}}Think step by step."""RED_FLAGS_AGENT_PROMPT = """You are a Red Flags Detector Agent evaluating candidate-job compatibility.**Inputs:**- Candidate full profile: {candidate}- Job full profile: {job}- Job's potential red flags: {job_red_flags}**Your task:**Identify red flags step-by-step:1. **Job-Defined Red Flags**: Check if candidate matches any red flags listed in job profile2. **Experience Red Flags**: Identify gaps, job hopping, lack of progression3. **Qualification Red Flags**: Over/under-qualification issues4. **Sector Red Flags**: Complete sector mismatches5. **Skill Red Flags**: Missing critical must-have skills6. **Location Red Flags**: Geographic incompatibility (if job requires relocation)7. **Criticality Assessment**: Classify red flags as critical vs minor**Critical Red Flags (auto-disqualify):**- Complete sector mismatch- Missing mandatory certifications/licenses- <50% of required experience years- Multiple job-defined red flags present**Minor Red Flags (reduce score but not disqualify):**- Over-qualification concerns- Minor skill gaps- Limited international experience (if preferred)**Scoring:**- No red flags: 100- Minor flags only: 70-90 (depending on number)- Critical flags present: 0-30**Output format (JSON):**{{  "agent_name": "red_flags_detector",  "score": <0-100>,  "alignment_level": "<perfect_match|strong_match|partial_match|weak_match|mismatch>",  "reasoning": "<step-by-step explanation>",  "evidence": [<red flag descriptions>],  "red_flags": [<all red flags with criticality: "critical" or "minor">],  "confidence": <0.0-1.0>}}Be thorough but fair. Think step by step."""DOMAIN_EXPERT_PROMPT = """You are a Domain Expert Agent for the {sector} sector.**Inputs:**- Candidate profile: {candidate}- Job profile: {job}- Previous evaluations: {previous_evaluations}**Your task:**Provide sector-specific deep evaluation:1. **Sector-Specific Skills**: Evaluate specialized skills critical for {sector}2. **Domain Experience**: Assess quality and relevance of candidate's experience in {sector}3. **Industry Patterns**: Check if career trajectory matches typical {sector} patterns4. **Certifications/Credentials**: Verify sector-specific requirements5. **Cultural Fit**: Assess if candidate's background aligns with {sector} culture6. **Validation**: Validate or challenge previous agents' evaluations with domain expertise7. **Scoring**: Provide sector-expert score**Sector-Specific Considerations for {sector}:**{sector_considerations}**Output format (JSON):**{{  "agent_name": "domain_expert_{sector}",  "score": <0-100>,  "alignment_level": "<perfect_match|strong_match|partial_match|weak_match|mismatch>",  "reasoning": "<sector-specific analysis>",  "evidence": [<domain-specific evidence>],  "red_flags": [<domain-specific red flags>],  "confidence": <0.0-1.0>}}Provide expert-level sector analysis."""ORCHESTRATOR_PROMPT = """You are the Orchestrator Agent synthesizing all evaluations.**Inputs:**- Sector evaluation: {sector_eval}- Skills evaluation: {skills_eval}- Seniority evaluation: {seniority_eval}- Competencies evaluation: {competencies_eval}- Red flags evaluation: {red_flags_eval}- Domain expert evaluation: {domain_expert_eval}- Weights: {weights}**Your task:**Synthesize final evaluation step-by-step:1. **Score Aggregation**: Apply weights to individual scores2. **Red Flags Integration**: Adjust score based on red flags3. **Consistency Check**: Validate that evaluations are consistent4. **Strengths Identification**: Extract top 3-5 strengths5. **Concerns Identification**: Extract top 3-5 concerns6. **Missing Skills**: List critical missing skills7. **Recommendation**: Make final recommendation8. **Decision Tree**: Create human-readable explanation**Weighted Score Calculation:**- overall_score = Σ(agent_score × weight)- If critical red flags present: cap overall_score at 40**Recommendation Thresholds:**- highly_recommend: ≥85- recommend: ≥70- acceptable: ≥55- not_recommend: <55 OR critical red flags**Output format (JSON):**{{  "overall_score": <0-100>,  "recommendation": "<highly_recommend|recommend|acceptable|not_recommend>",  "summary": "<2-3 sentence executive summary>",  "strengths": [<3-5 key strengths>],  "concerns": [<3-5 key concerns>],  "missing_skills": [<critical missing skills>],  "all_red_flags": [<all red flags from all agents>],  "agent_scores": {{    "sector": <score>,    "skills": <score>,    "seniority": <score>,    "competencies": <score>,    "red_flags": <score>,    "domain_expert": <score>  }},  "decision_tree": "<human-readable step-by-step decision explanation>"}}Be comprehensive and precise. Provide full transparency."""# Sector-specific considerationsSECTOR_CONSIDERATIONS = {    "construction": """- Prioritize: Site experience, safety certifications (PRL in Spain), subcontractor management- Check: Infrastructure types (roads, buildings, bridges), project scale, international exposure- Red flags: Only office experience, no field work, lack of execution experience""",    "software": """- Prioritize: Technical stack match, system design experience, agile practices- Check: Open source contributions, scalability experience, architecture patterns- Red flags: Outdated tech stack, no CI/CD experience, lack of testing practices""",    "finance": """- Prioritize: Regulatory knowledge, risk management, financial modeling- Check: Certifications (CFA, FRM), compliance experience, audit exposure- Red flags: No regulatory experience, lack of financial analysis skills""",    "healthcare": """- Prioritize: Clinical experience, healthcare regulations, patient safety- Check: Licenses, certifications, EHR systems experience- Red flags: Expired licenses, no patient-facing experience for clinical roles""",}

## Agent Implementations

Each specialized agent with error handling and structured outputs.

In [ ]:
%%writefile ../src/candidate_job_evaluation/agents.py"""Specialized evaluation agents."""import jsonfrom typing import Any, Dict, Optionalfrom langchain_core.messages import HumanMessagefrom langchain_openai import ChatOpenAIfrom .config import EvaluationConfigfrom .prompts import (    COMPETENCIES_AGENT_PROMPT,    DOMAIN_EXPERT_PROMPT,    ORCHESTRATOR_PROMPT,    RED_FLAGS_AGENT_PROMPT,    SECTOR_AGENT_PROMPT,    SECTOR_CONSIDERATIONS,    SENIORITY_AGENT_PROMPT,    SKILLS_AGENT_PROMPT,)from .state import AgentEvaluation, EvaluationState, FinalEvaluationclass EvaluationAgents:    """Container for all specialized evaluation agents."""    def __init__(self, config: EvaluationConfig):        """Initialize agents with configuration.        Args:            config: Evaluation configuration        """        self.config = config        self.llm = ChatOpenAI(            model=config.llm_model,            temperature=config.llm_temperature,        )    def _safe_agent_call(        self, prompt: str, agent_name: str, state: EvaluationState    ) -> AgentEvaluation:        """Safely call an agent with error handling.        Args:            prompt: Formatted prompt for the agent            agent_name: Name of the agent            state: Current evaluation state        Returns:            AgentEvaluation with results or fallback        """        try:            response = self.llm.invoke([HumanMessage(content=prompt)])            result = json.loads(response.content)            return AgentEvaluation(**result)        except Exception as e:            print(f"Error in {agent_name}: {e}")            # Fallback evaluation            return AgentEvaluation(                agent_name=agent_name,                score=self.config.agent_failure_default_score,                alignment_level="partial_match",                reasoning=f"Agent failed with error: {str(e)}. Using default score.",                evidence=[],                red_flags=[f"agent_error: {agent_name}"],                confidence=0.3,            )    def sector_alignment_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Evaluate sector alignment between candidate and job.        Args:            state: Current evaluation state        Returns:            Updated state with sector evaluation        """        candidate = state["candidate"]        job = state["job_offer"]        prompt = SECTOR_AGENT_PROMPT.format(            candidate_sector=json.dumps(                candidate["semantic_enrichment"]["analysis_components"][                    "sector_inference"                ],                indent=2,            ),            job_sector=json.dumps(                job["semantic_enrichment"]["analysis_components"]["sector_inference"],                indent=2,            ),            candidate_disambiguation=json.dumps(                candidate["semantic_enrichment"]["disambiguation_map"], indent=2            ),            job_red_flags=json.dumps(                job["semantic_enrichment"]["ideal_candidate"].get(                    "potential_red_flags", []                ),                indent=2,            ),        )        evaluation = self._safe_agent_call(prompt, "sector_alignment", state)        return {            "sector_evaluation": evaluation,            "messages": [                HumanMessage(content=f"Sector alignment evaluated: {evaluation.score}/100")            ],        }    def skills_matching_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Evaluate skills match between candidate and job.        Args:            state: Current evaluation state        Returns:            Updated state with skills evaluation        """        candidate = state["candidate"]        job = state["job_offer"]        prompt = SKILLS_AGENT_PROMPT.format(            candidate_skills_explicit=json.dumps(                candidate["semantic_enrichment"]["structured_skills"]["explicit"],                indent=2,            ),            candidate_skills_implicit=json.dumps(                candidate["semantic_enrichment"]["structured_skills"]["implicit"],                indent=2,            ),            job_required_skills=json.dumps(                [                    s                    for s in job["semantic_enrichment"]["structured_skills"].get(                        "explicit", []                    )                    + job["semantic_enrichment"]["structured_skills"].get(                        "implicit", []                    )                    if s.get("importance") == "required"                ],                indent=2,            ),            job_preferred_skills=json.dumps(                [                    s                    for s in job["semantic_enrichment"]["structured_skills"].get(                        "explicit", []                    )                    + job["semantic_enrichment"]["structured_skills"].get(                        "implicit", []                    )                    if s.get("importance") == "preferred"                ],                indent=2,            ),            candidate_synonyms=json.dumps(                candidate["semantic_enrichment"]["synonym_map"], indent=2            ),            job_must_have=json.dumps(                job["semantic_enrichment"]["ideal_candidate"].get(                    "must_have_experience", []                ),                indent=2,            ),        )        evaluation = self._safe_agent_call(prompt, "skills_matching", state)        return {            "skills_evaluation": evaluation,            "messages": [                HumanMessage(content=f"Skills matching evaluated: {evaluation.score}/100")            ],        }    def seniority_alignment_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Evaluate seniority alignment between candidate and job.        Args:            state: Current evaluation state        Returns:            Updated state with seniority evaluation        """        candidate = state["candidate"]        job = state["job_offer"]        prompt = SENIORITY_AGENT_PROMPT.format(            candidate_seniority=candidate["semantic_enrichment"][                "analysis_components"            ]["seniority_inference"]["seniority_level"],            candidate_years=candidate["semantic_enrichment"]["analysis_components"][                "seniority_inference"            ]["years_experience_estimate"],            job_seniority=job["semantic_enrichment"]["analysis_components"][                "seniority_inference"            ]["required_seniority"],            job_years=job["semantic_enrichment"]["analysis_components"][                "seniority_inference"            ]["years_experience_required"],            candidate_seniority_evidence=json.dumps(                candidate["semantic_enrichment"]["analysis_components"][                    "seniority_inference"                ]["evidence"],                indent=2,            ),            job_red_flags=json.dumps(                job["semantic_enrichment"]["ideal_candidate"].get(                    "potential_red_flags", []                ),                indent=2,            ),        )        evaluation = self._safe_agent_call(prompt, "seniority_alignment", state)        return {            "seniority_evaluation": evaluation,            "messages": [                HumanMessage(                    content=f"Seniority alignment evaluated: {evaluation.score}/100"                )            ],        }    def competencies_matching_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Evaluate competencies match between candidate and job.        Args:            state: Current evaluation state        Returns:            Updated state with competencies evaluation        """        candidate = state["candidate"]        job = state["job_offer"]        prompt = COMPETENCIES_AGENT_PROMPT.format(            candidate_competencies=json.dumps(                candidate["semantic_enrichment"]["structured_competencies"], indent=2            ),            job_competencies=json.dumps(                job["semantic_enrichment"]["structured_competencies"], indent=2            ),            candidate_leadership=json.dumps(                candidate["semantic_enrichment"]["analysis_components"][                    "competencies_analysis"                ].get("leadership_indicators", []),                indent=2,            ),            job_leadership_level=job["semantic_enrichment"]["analysis_components"][                "competencies_analysis"            ].get("leadership_level", "individual_contributor"),        )        evaluation = self._safe_agent_call(prompt, "competencies_matching", state)        return {            "competencies_evaluation": evaluation,            "messages": [                HumanMessage(                    content=f"Competencies matching evaluated: {evaluation.score}/100"                )            ],        }    def red_flags_detector_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Detect red flags in candidate-job match.        Args:            state: Current evaluation state        Returns:            Updated state with red flags evaluation        """        candidate = state["candidate"]        job = state["job_offer"]        prompt = RED_FLAGS_AGENT_PROMPT.format(            candidate=json.dumps(candidate, indent=2),            job=json.dumps(job, indent=2),            job_red_flags=json.dumps(                job["semantic_enrichment"]["ideal_candidate"].get(                    "potential_red_flags", []                ),                indent=2,            ),        )        evaluation = self._safe_agent_call(prompt, "red_flags_detector", state)        # Check for critical red flags        critical_flags = [            flag            for flag in evaluation.red_flags            if any(                critical_type in flag.lower()                for critical_type in self.config.critical_red_flag_types            )        ]        critical_red_flags_found = len(critical_flags) > 0        return {            "red_flags_evaluation": evaluation,            "critical_red_flags_found": critical_red_flags_found,            "early_termination": critical_red_flags_found,            "messages": [                HumanMessage(                    content=f"Red flags detection: {len(evaluation.red_flags)} flags found ({'CRITICAL' if critical_red_flags_found else 'MINOR'})"                )            ],        }    def domain_expert_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Provide domain-specific expert evaluation.        Args:            state: Current evaluation state        Returns:            Updated state with domain expert evaluation        """        job = state["job_offer"]        sector = job["semantic_enrichment"]["analysis_components"]["sector_inference"][            "primary_sector"        ]        sector_considerations = SECTOR_CONSIDERATIONS.get(            sector, "No specific considerations for this sector."        )        previous_evaluations = {            "sector": state.get("sector_evaluation"),            "skills": state.get("skills_evaluation"),            "seniority": state.get("seniority_evaluation"),            "competencies": state.get("competencies_evaluation"),            "red_flags": state.get("red_flags_evaluation"),        }        prompt = DOMAIN_EXPERT_PROMPT.format(            sector=sector,            candidate=json.dumps(state["candidate"], indent=2),            job=json.dumps(job, indent=2),            previous_evaluations=json.dumps(                {                    k: v.model_dump() if v else None                    for k, v in previous_evaluations.items()                },                indent=2,            ),            sector_considerations=sector_considerations,        )        evaluation = self._safe_agent_call(prompt, f"domain_expert_{sector}", state)        return {            "domain_expert_evaluation": evaluation,            "messages": [                HumanMessage(                    content=f"Domain expert ({sector}) evaluated: {evaluation.score}/100"                )            ],        }    def orchestrator_agent(self, state: EvaluationState) -> Dict[str, Any]:        """Synthesize all evaluations into final verdict.        Args:            state: Current evaluation state        Returns:            Updated state with final evaluation        """        job = state["job_offer"]        sector = job["semantic_enrichment"]["analysis_components"]["sector_inference"][            "primary_sector"        ]        # Get weights (apply sector-specific adjustments)        weights = self.config.default_weights.copy()        if sector in self.config.sector_weight_adjustments:            weights.update(self.config.sector_weight_adjustments[sector])        # Normalize weights to sum to 1        total_weight = sum(weights.values())        weights = {k: v / total_weight for k, v in weights.items()}        prompt = ORCHESTRATOR_PROMPT.format(            sector_eval=json.dumps(                state["sector_evaluation"].model_dump()                if state.get("sector_evaluation")                else None,                indent=2,            ),            skills_eval=json.dumps(                state["skills_evaluation"].model_dump()                if state.get("skills_evaluation")                else None,                indent=2,            ),            seniority_eval=json.dumps(                state["seniority_evaluation"].model_dump()                if state.get("seniority_evaluation")                else None,                indent=2,            ),            competencies_eval=json.dumps(                state["competencies_evaluation"].model_dump()                if state.get("competencies_evaluation")                else None,                indent=2,            ),            red_flags_eval=json.dumps(                state["red_flags_evaluation"].model_dump()                if state.get("red_flags_evaluation")                else None,                indent=2,            ),            domain_expert_eval=json.dumps(                state["domain_expert_evaluation"].model_dump()                if state.get("domain_expert_evaluation")                else None,                indent=2,            ),            weights=json.dumps(weights, indent=2),        )        try:            response = self.llm.invoke([HumanMessage(content=prompt)])            result = json.loads(response.content)            final_evaluation = FinalEvaluation(**result)        except Exception as e:            print(f"Error in orchestrator: {e}")            # Fallback: calculate simple weighted average            scores = {                "sector": state.get("sector_evaluation").score                if state.get("sector_evaluation")                else 50,                "skills": state.get("skills_evaluation").score                if state.get("skills_evaluation")                else 50,                "seniority": state.get("seniority_evaluation").score                if state.get("seniority_evaluation")                else 50,                "competencies": state.get("competencies_evaluation").score                if state.get("competencies_evaluation")                else 50,                "domain_expert": state.get("domain_expert_evaluation").score                if state.get("domain_expert_evaluation")                else 50,            }            overall_score = int(                sum(scores.get(k, 50) * weights.get(k, 0.2) for k in weights.keys())            )            if state.get("critical_red_flags_found"):                overall_score = min(overall_score, 40)            if overall_score >= self.config.highly_recommend_threshold:                recommendation = "highly_recommend"            elif overall_score >= self.config.recommend_threshold:                recommendation = "recommend"            elif overall_score >= self.config.acceptable_threshold:                recommendation = "acceptable"            else:                recommendation = "not_recommend"            final_evaluation = FinalEvaluation(                overall_score=overall_score,                recommendation=recommendation,                summary=f"Orchestrator failed. Fallback score: {overall_score}/100. Error: {str(e)}",                strengths=[],                concerns=["Orchestrator agent failed"],                missing_skills=[],                all_red_flags=[],                agent_scores=scores,                decision_tree="Fallback evaluation due to orchestrator failure.",            )        return {            "final_evaluation": final_evaluation,            "messages": [                HumanMessage(                    content=f"Final evaluation: {final_evaluation.overall_score}/100 - {final_evaluation.recommendation}"                )            ],        }

## Graph Construction

Building the LangGraph workflow with conditional edges.

In [ ]:
%%writefile ../src/candidate_job_evaluation/graph.py"""LangGraph workflow for candidate-job evaluation."""from datetime import datetimefrom typing import Literalfrom uuid import uuid4from langgraph.graph import END, START, StateGraphfrom .agents import EvaluationAgentsfrom .config import EvaluationConfigfrom .state import EvaluationStatedef should_terminate_early(state: EvaluationState) -> Literal["terminate", "continue"]:    """Decide if evaluation should terminate early due to critical red flags.    Args:        state: Current evaluation state    Returns:        "terminate" if critical red flags found, else "continue"    """    if state.get("early_termination", False):        return "terminate"    return "continue"def build_evaluation_graph(config: EvaluationConfig) -> StateGraph:    """Build the candidate-job evaluation graph.    Architecture:    1. Parallel execution of: sector, skills, seniority, competencies, red_flags    2. Conditional edge: if critical red flags → terminate early    3. Domain expert evaluation (sector-specific)    4. Orchestrator synthesis → final evaluation    Args:        config: Evaluation configuration    Returns:        Compiled StateGraph    """    agents = EvaluationAgents(config)    # Initialize graph    graph = StateGraph(EvaluationState)    # Add specialized agent nodes    graph.add_node("sector_alignment", agents.sector_alignment_agent)    graph.add_node("skills_matching", agents.skills_matching_agent)    graph.add_node("seniority_alignment", agents.seniority_alignment_agent)    graph.add_node("competencies_matching", agents.competencies_matching_agent)    graph.add_node("red_flags_detector", agents.red_flags_detector_agent)    # Add domain expert and orchestrator nodes    graph.add_node("domain_expert", agents.domain_expert_agent)    graph.add_node("orchestrator", agents.orchestrator_agent)    # Phase 1: Parallel specialized analysis    graph.add_edge(START, "sector_alignment")    graph.add_edge(START, "skills_matching")    graph.add_edge(START, "seniority_alignment")    graph.add_edge(START, "competencies_matching")    graph.add_edge(START, "red_flags_detector")    # Phase 2: Conditional routing after red flags detection    # If critical red flags found, skip domain expert and go to orchestrator    # Otherwise, continue to domain expert    graph.add_conditional_edges(        "red_flags_detector",        should_terminate_early,        {"terminate": "orchestrator", "continue": "domain_expert"},    )    # Wait for all parallel agents before domain expert    graph.add_edge("sector_alignment", "domain_expert")    graph.add_edge("skills_matching", "domain_expert")    graph.add_edge("seniority_alignment", "domain_expert")    graph.add_edge("competencies_matching", "domain_expert")    # Phase 3: Final synthesis    graph.add_edge("domain_expert", "orchestrator")    graph.add_edge("orchestrator", END)    return graph.compile()def evaluate_candidate_for_job(    candidate: dict, job_offer: dict, config: EvaluationConfig = None) -> dict:    """Evaluate a candidate for a job offer.    Args:        candidate: Enriched candidate profile        job_offer: Enriched job offer profile        config: Evaluation configuration (uses default if None)    Returns:        Final evaluation with all agent results    """    if config is None:        config = EvaluationConfig()    graph = build_evaluation_graph(config)    initial_state: EvaluationState = {        "candidate": candidate,        "job_offer": job_offer,        "sector_evaluation": None,        "skills_evaluation": None,        "seniority_evaluation": None,        "competencies_evaluation": None,        "red_flags_evaluation": None,        "domain_expert_evaluation": None,        "final_evaluation": None,        "critical_red_flags_found": False,        "early_termination": False,        "evaluation_id": str(uuid4()),        "timestamp": datetime.now().isoformat(),        "messages": [],    }    result = graph.invoke(initial_state)    return result

## Example Usage

Demonstrating the evaluation system with test data.

In [ ]:
# Load test data
import json
from pathlib import Path

test_dir = Path("../tests")

# Load Natalia (candidate)
with open(test_dir / "candidates" / "outputs" / "enriched_natalia.json") as f:
    candidate_natalia = json.load(f)

# Load Acciona job (construction sector)
with open(test_dir / "jobs" / "outputs" / "enriched_acciona_job.json") as f:
    job_acciona = json.load(f)

print("Loaded test data:")
print(f"- Candidate: {candidate_natalia['perfil_raw']['full_name']}")
print(f"- Job: {job_acciona['raw_job_posting']['job_title']} at {job_acciona['raw_job_posting']['company_name']}")

In [ ]:
# Run evaluation
from candidate_job_evaluation.graph import evaluate_candidate_for_job
from candidate_job_evaluation.config import EvaluationConfig

# Create config
config = EvaluationConfig(
    llm_model="gpt-4o",
    llm_temperature=0.1,
)

print("\n" + "="*80)
print("RUNNING EVALUATION: Natalia López Andrés vs Acciona Construction Manager")
print("="*80 + "\n")

result = evaluate_candidate_for_job(
    candidate=candidate_natalia,
    job_offer=job_acciona,
    config=config
)

# Display results
final_eval = result["final_evaluation"]

print("\n" + "="*80)
print("FINAL EVALUATION RESULTS")
print("="*80)
print(f"\nOverall Score: {final_eval.overall_score}/100")
print(f"Recommendation: {final_eval.recommendation.upper()}")
print(f"\nSummary:\n{final_eval.summary}")

print(f"\nAgent Scores:")
for agent, score in final_eval.agent_scores.items():
    print(f"  - {agent}: {score}/100")

print(f"\nStrengths ({len(final_eval.strengths)}):")
for i, strength in enumerate(final_eval.strengths, 1):
    print(f"  {i}. {strength}")

print(f"\nConcerns ({len(final_eval.concerns)}):")
for i, concern in enumerate(final_eval.concerns, 1):
    print(f"  {i}. {concern}")

if final_eval.missing_skills:
    print(f"\nMissing Skills ({len(final_eval.missing_skills)}):")
    for i, skill in enumerate(final_eval.missing_skills, 1):
        print(f"  {i}. {skill}")

if final_eval.all_red_flags:
    print(f"\nRed Flags ({len(final_eval.all_red_flags)}):")
    for i, flag in enumerate(final_eval.all_red_flags, 1):
        print(f"  {i}. {flag}")

print(f"\nDecision Tree:\n{final_eval.decision_tree}")
print("\n" + "="*80)

## Save Results

Persist evaluation results for analysis.

In [ ]:
# Save evaluation result
output_dir = Path("../tests/evaluations/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "evaluation_natalia_acciona.json"

# Convert to serializable format
result_serializable = {
    "evaluation_id": result["evaluation_id"],
    "timestamp": result["timestamp"],
    "candidate_id": candidate_natalia["candidato_id"],
    "job_id": job_acciona["job_id"],
    "sector_evaluation": result["sector_evaluation"].model_dump() if result.get("sector_evaluation") else None,
    "skills_evaluation": result["skills_evaluation"].model_dump() if result.get("skills_evaluation") else None,
    "seniority_evaluation": result["seniority_evaluation"].model_dump() if result.get("seniority_evaluation") else None,
    "competencies_evaluation": result["competencies_evaluation"].model_dump() if result.get("competencies_evaluation") else None,
    "red_flags_evaluation": result["red_flags_evaluation"].model_dump() if result.get("red_flags_evaluation") else None,
    "domain_expert_evaluation": result["domain_expert_evaluation"].model_dump() if result.get("domain_expert_evaluation") else None,
    "final_evaluation": result["final_evaluation"].model_dump() if result.get("final_evaluation") else None,
}

with open(output_file, "w") as f:
    json.dump(result_serializable, f, indent=2)

print(f"\nEvaluation saved to: {output_file}")